# 2. Full-prefix probability model

This notebook estimates the probability of the next token using **the complete prefix since the beginning of the sentence**, rather than a fixed-size N-gram.

For a prefix (history) \(h\), the estimate is:

\[
P(w_{t+1}\mid h)=\frac{C(h,w_{t+1})}{C(h)}
\]

This is sometimes described as a **variable-order Markov model** or **full-history (unbounded-history) model**. In general NLP, storing and estimating every distinct prefix is impractical because the number of possible histories grows very rapidly. Our restricted fictional language makes it possible to demonstrate the idea directly.

The notebook reads `observations.csv`, learns the prefix probabilities, saves them to `prefix-probabilities.csv`, and generates a sentence one token at a time by sampling from the learned distribution.


In [1]:
import random
from collections import Counter, defaultdict
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

INPUT_FILE = "observations.csv"
OUTPUT_FILE = "probabilities-prefix.csv"

# Initial prefix used by the generator.
START = "s"

# Set to an integer for reproducible generation, or None
# to obtain potentially different results on each run.
RANDOM_SEED = None

MAX_GENERATED_TOKENS = 20

if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)


In [2]:
# ------------------------------------------------------------
# Read observations
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print(f"Read {len(df)} observations.")
df.head()


Read 850 observations.


,id,sentence
0,1,s f e
1,2,s l l f e
2,3,s f e
3,4,s r r f e
4,5,s f e


## Count every prefix → next-token combination

For each sentence we add `<s>` as a beginning-of-sentence marker and `</s>` as an end marker.

For example, the observation

```text
s r r f e
```

becomes

```text
<s> s r r f e </s>
```

and contributes the following full-prefix observations:

```text
<s>                 -> s
<s> s               -> r
<s> s r             -> r
<s> s r r           -> f
<s> s r r f         -> e
<s> s r r f e       -> </s>
```

Unlike an N-gram model, the history grows by one token at every step.


In [3]:
def tokenize(sentence):
    return sentence.split()

prefix_next_counts = Counter()
prefix_counts = Counter()

for sentence in df["sentence"]:
    tokens = ["<s>"] + tokenize(sentence) + ["</s>"]

    # Every position contributes one (complete prefix, next token) pair.
    for i in range(len(tokens) - 1):
        history = tuple(tokens[:i + 1])
        next_token = tokens[i + 1]

        prefix_next_counts[(history, next_token)] += 1
        prefix_counts[history] += 1

print(f"Distinct prefix -> next-token combinations: {len(prefix_next_counts)}")
print(f"Distinct prefixes: {len(prefix_counts)}")


Distinct prefix -> next-token combinations: 56
Distinct prefixes: 40


## Compute prefix probabilities

For every observed pair `(history, next_token)`:

\[
P(\text{next\_token}\mid\text{history})
=
\frac{\text{combination count}}{\text{history count}}
\]

Here `combination_count` means the number of observations in which that **exact complete prefix** was followed by that token. `history_count` is the number of observations reaching that prefix.


In [4]:
prefix_probabilities = []

for (history, next_token), combination_count in prefix_next_counts.items():
    history_count = prefix_counts[history]
    probability = combination_count / history_count

    prefix_probabilities.append({
        "history": " ".join(history),
        "next_token": next_token,
        "combination_count": combination_count,
        "history_count": history_count,
        "probability": probability,
    })

prefix_df = (
    pd.DataFrame(prefix_probabilities)
    .sort_values(["history", "probability", "next_token"],
                ascending=[True, False, True])
    .reset_index(drop=True)
)

prefix_df


,history,next_token,combination_count,history_count,probability
0,<s>,s,763,850,0.897647
1,<s>,i,87,850,0.102353
2,<s> i,l,87,87,1.000000
3,<s> i l,l,87,87,1.000000
4,<s> i l l,l,50,87,0.574713
5,<s> i l l,</s>,37,87,0.425287
6,<s> i l l l,l,50,50,1.000000
7,<s> i l l l l,</s>,50,50,1.000000
8,<s> s,l,306,763,0.401048
9,<s> s,r,304,763,0.398427


In [5]:
# ------------------------------------------------------------
# Save prefix probabilities
# ------------------------------------------------------------

prefix_df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved {len(prefix_df)} prefix probability records to: {OUTPUT_FILE}")


Saved 56 prefix probability records to: probabilities-prefix.csv


## Generate a sentence step by step

The generator uses the **entire generated prefix** as its state. Therefore, unlike the N-gram notebook, the state is not truncated to the last `N-1` tokens.


In [6]:
# Build transitions for sampling.
prefix_transitions = defaultdict(list)

for row in prefix_df.itertuples():
    history = tuple(row.history.split())
    prefix_transitions[history].append(
        (row.next_token, row.probability)
    )


def sample_next(history):
    options = prefix_transitions.get(tuple(history))

    if not options:
        return None

    tokens = [token for token, _ in options]
    probabilities = [probability for _, probability in options]

    return random.choices(tokens, weights=probabilities, k=1)[0]


def generate_from_start(start, max_tokens=20, verbose=True):
    """Generate a continuation using the complete prefix as history."""
    generated = tokenize(start)

    if not generated:
        raise ValueError("START must contain at least one token.")

    # Include <s> in the history, because it was used when learning.
    history = ["<s>"] + generated

    for step in range(1, max_tokens + 1):
        next_token = sample_next(history)

        if verbose:
            print(
                f"Step {step:2d}: "
                f"history={tuple(history)} -> next={next_token}"
            )

        if next_token is None:
            print("No transition was observed for this complete prefix.")
            break

        if next_token == "</s>":
            break

        generated.append(next_token)
        history.append(next_token)

    return " ".join(generated)


In [7]:
# ------------------------------------------------------------
# Generate one sentence
# ------------------------------------------------------------

print("Generated sentence:")
result = generate_from_start(
    START,
    max_tokens=MAX_GENERATED_TOKENS,
    verbose=True,
)

print("\nFinal:", result)


Generated sentence:
Step  1: history=('<s>', 's') -> next=l
Step  2: history=('<s>', 's', 'l') -> next=l
Step  3: history=('<s>', 's', 'l', 'l') -> next=p
Step  4: history=('<s>', 's', 'l', 'l', 'p') -> next=e
Step  5: history=('<s>', 's', 'l', 'l', 'p', 'e') -> next=</s>

Final: s l l p e


## Compare the idea with the N-gram model

| Model | History used to predict the next token |
|---|---|
| Bigram | 1 previous token |
| Trigram | 2 previous tokens |
| N-gram | `N-1` previous tokens |
| Full-prefix model | **All tokens from the beginning** |

The full-prefix model therefore uses a different history length at each position. It can be understood as a variable-order / full-history Markov model: the complete prefix is the state.

For this fictional language, the restricted grammar means that the number of possible prefixes remains small enough for the demonstration. For unrestricted natural language, the number of distinct prefixes is enormous, which is why practical N-gram models limit the history and modern neural language models use compressed representations of context instead.
